# Solution — Task 02: Few-Shot Entity Extraction

## Setup

In [ ]:
from openai import OpenAI
import json, os

# SET YOUR API KEY HERE
api_key = "your-api-key-here"
client = OpenAI(api_key=api_key)

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

with open(os.path.join(FIXTURES, "extraction_samples.json")) as f:
    extraction_samples = json.load(f)

def parse_json_safe(text: str) -> dict | None:
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        text = "\n".join(lines[1:-1])
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

print(f"✓ Setup complete. {len(extraction_samples)} extraction samples loaded.")

## Solution 2.1 — FEW_SHOT_EXAMPLES

In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "input": "I bought a MacBook Pro M3 last week and the keyboard stopped working after the OS update.",
        "output": '{"product": "MacBook Pro M3", "issue": "keyboard not working", "sentiment": "negative"}',
    },
    {
        "input": "The Spotify Premium subscription I purchased yesterday is not showing as active.",
        "output": '{"product": "Spotify Premium", "issue": "subscription not activating", "sentiment": "negative"}',
    },
    {
        "input": "The Samsung Galaxy S24 works flawlessly with your integration. Excellent!",
        "output": '{"product": "Samsung Galaxy S24", "issue": null, "sentiment": "positive"}',
    },
    {
        "input": "My Dell XPS 15 arrived with a cracked screen. I need a replacement immediately.",
        "output": '{"product": "Dell XPS 15", "issue": "cracked screen on arrival", "sentiment": "negative"}',
    },
    {
        "input": "The Logitech MX Master 3 mouse is exactly as described. Very satisfied.",
        "output": '{"product": "Logitech MX Master 3", "issue": null, "sentiment": "positive"}',
    },
]

assert len(FEW_SHOT_EXAMPLES) >= 3
for i, ex in enumerate(FEW_SHOT_EXAMPLES):
    assert "input" in ex and "output" in ex
    parsed = json.loads(ex["output"])
    assert "product" in parsed and "sentiment" in parsed
    assert parsed["sentiment"] in ("positive", "negative")
sentiments = [json.loads(ex["output"])["sentiment"] for ex in FEW_SHOT_EXAMPLES]
assert "positive" in sentiments and "negative" in sentiments
assert sum(1 for ex in FEW_SHOT_EXAMPLES if json.loads(ex["output"]).get("issue") is None) >= 1
print(f"✓ Task 2.1 passed ({len(FEW_SHOT_EXAMPLES)} examples)")

## Solution 2.2 — EXTRACTION_SYSTEM_PROMPT

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are an entity extractor for customer messages.

Extract these fields from each message:
- product: the specific product or service mentioned
- issue: the problem reported (set to null if no problem — customer is happy)
- sentiment: "positive" if customer is satisfied, "negative" if reporting a problem

Return ONLY valid JSON:
{"product": "...", "issue": "..." or null, "sentiment": "positive|negative"}
"""

assert len(EXTRACTION_SYSTEM_PROMPT.strip()) >= 50
assert all(k in EXTRACTION_SYSTEM_PROMPT.lower() for k in ["product", "sentiment", "json"])
assert "positive" in EXTRACTION_SYSTEM_PROMPT.lower() and "negative" in EXTRACTION_SYSTEM_PROMPT.lower()
print("✓ Task 2.2 passed")

## Solution 2.3 — build_extraction_prompt()

In [ ]:
def build_extraction_prompt(text: str, examples: list[dict]) -> str:
    lines = []
    for ex in examples:
        lines.append(f"Message: {ex['input']}")
        lines.append(f"Output: {ex['output']}")
        lines.append("")
    lines.append(f"Message: {text}")
    lines.append("Output:")
    return "\n".join(lines)

test_text = "My Sony WF-1000XM5 earbuds have terrible battery life."
prompt = build_extraction_prompt(test_text, FEW_SHOT_EXAMPLES)
assert test_text in prompt
assert len(prompt) > len(test_text)
for ex in FEW_SHOT_EXAMPLES[:2]:
    assert ex["input"] in prompt
print("✓ Task 2.3 passed")

## Solution 2.4 — extract_entities()

In [ ]:
def extract_entities(client, text: str) -> dict:
    user_prompt = build_extraction_prompt(text, FEW_SHOT_EXAMPLES)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.0,
    )
    return json.loads(response.choices[0].message.content)

sample = extraction_samples[0]
result = extract_entities(client, sample["text"])
assert isinstance(result, dict) and "product" in result and "sentiment" in result
assert result["sentiment"] in ("positive", "negative")
assert isinstance(result.get("product"), str) and len(result["product"]) > 0
print(f"✓ Task 2.4 passed — Got: {result}")

positive_sample = extraction_samples[2]
result_pos = extract_entities(client, positive_sample["text"])
assert result_pos.get("sentiment") == "positive"
assert result_pos.get("issue") is None or result_pos.get("issue") == ""
print("✓ Task 2.4b passed")

## Solution 2.5 — Format Accuracy

In [ ]:
format_ok = 0
sentiment_correct = 0

for s in extraction_samples:
    result = extract_entities(client, s["text"])
    has_keys = all(k in result for k in ["product", "sentiment"])
    valid_sentiment = result.get("sentiment") in ("positive", "negative")
    nonempty_product = isinstance(result.get("product"), str) and len(result["product"]) > 0
    if has_keys and valid_sentiment and nonempty_product:
        format_ok += 1
    if result.get("sentiment") == s["expected"]["sentiment"]:
        sentiment_correct += 1
    print(f"  {result.get('product')!r:30} {result.get('sentiment'):10} (expected {s['expected']['sentiment']})")

format_rate = format_ok / len(extraction_samples)
sentiment_acc = sentiment_correct / len(extraction_samples)
print(f"\nFormat: {format_rate:.0%} | Sentiment accuracy: {sentiment_acc:.0%}")
assert format_rate == 1.0
assert sentiment_acc >= 0.80
print("✓ Task 2.5 passed")

## Done

In [ ]:
print("\n✓ All task_02 tests passed!")